In [49]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ortools.linear_solver import pywraplp

## Clase 5

<hr>

## C05E1 — Transporte balanceado

### Situación completa

* SaludRed distribuye kits médicos estandarizados desde dos plantas a tres centros regionales durante la próxima semana. 

* La planta P0 puede despachar hasta 35 lotes y P1 hasta 25. 

* Los centros D0, D1 y D2 deben recibir exactamente 20, 25 y 15 lotes, respectivamente.

*  La oferta total de 60 coincide con la demanda total de 60, por lo que no hace falta inventario ni faltante en este caso. 

* Cada variable x_ij representa lotes enviados desde la planta i al centro j y se considera continua porque un lote de planificación agrega muchas unidades físicas divisibles. 

* El costo por lote depende de la ruta: desde P0 es 4, 6 y 8 hacia D0, D1 y D2; desde P1 es 5, 4 y 3.

* Los costos están en centenas de unidades monetarias y ya incluyen transporte y manipulación. 

* La responsable logística debe elegir los seis flujos para cumplir toda la demanda sin superar ninguna oferta y minimizar el costo semanal. 

* Los balances de destino expresan servicio obligatorio; las restricciones de planta expresan disponibilidad. 

* Esta distinción será esencial cuando más adelante se permitan inventario o faltantes.

### Pregunta de decisión

¿Cuántos lotes debe enviar cada planta a cada centro para cubrir exactamente las tres demandas al menor costo semanal?

In [50]:
# Desde dos plantas

# p0 despacha hasta 35 lotes

# p1 despacha hasta 25 lotes

# Centros

# d0 recibe 20 lotes

# d1 recibe 25 lotes

# d2 recibe 15 lotes

# Oferta total

# 60 coincide con la demanda de 60

# Costo por lote

#         D0 | D1 | D2
# p0 =>  4   | 6  | 8
# p1 =>  5   | 4  | 3

# Pregunta de decisión

# ¿Cuántos lotes debe enviar cada planta a cada centro para cubrir exactamente las tres demandas al menor costo semanal?

# -------------------------------------------------------------------------------------------------------------------------

In [51]:
variables = [
    ['p0-d0', 4, 35, 20],
    ['p0-d1', 6, 35, 25],
    ['p0-d2', 8, 35, 15],
    ['p1-d0', 5, 25, 20],
    ['p1-d1', 4, 25, 25],
    ['p1-d2', 3, 25, 15],
]

variables = pd.DataFrame(variables, columns=['arco', 'costo', 'oferta', 'demanda'])
variables

,arco,costo,oferta,demanda
0,p0-d0,4,35,20
1,p0-d1,6,35,25
2,p0-d2,8,35,15
3,p1-d0,5,25,20
4,p1-d1,4,25,25
5,p1-d2,3,25,15


In [52]:
# Contraints:

solver = pywraplp.Solver.CreateSolver("GLOP")

# Variables:
# Definición de variables de decisión para cada arco
x = {}
for idx, row in variables.iterrows():
    x[row['arco']] = solver.NumVar(0, row['oferta'], f"x_{row['arco']}")

# Restricciones:
solver.Add(x['p0-d0'] + x['p0-d1'] + x['p0-d2'] == 35, "planta 0")
solver.Add(x['p1-d0'] + x['p1-d1'] + x['p1-d2'] == 25, "planta 1")
solver.Add(x['p0-d0'] + x['p1-d0'] == 20, "demanda 0")
solver.Add(x['p0-d1'] + x['p1-d1'] == 25, "demanda 1")
solver.Add(x['p0-d2'] + x['p1-d2'] == 15, "demanda 2")

# Función objetivo:
solver.Minimize(x['p0-d0'] * variables.loc[0, 'costo'] + x['p0-d1'] * variables.loc[1, 'costo'] + x['p0-d2'] * variables.loc[2, 'costo'] + x['p1-d0'] * variables.loc[3, 'costo'] + x['p1-d1'] * variables.loc[4, 'costo'] + x['p1-d2'] * variables.loc[5, 'costo'])

# Print
status = solver.Solve()
es_optimo = status == pywraplp.Solver.OPTIMAL
print("Viable:", es_optimo)

if es_optimo:
    print("Solución óptima encontrada:")
    for arco in x:
        print(f"{arco}: {x[arco].solution_value():.2f}")
    print("Costo mínimo total =", solver.Objective().Value())

Viable: True
Solución óptima encontrada:
p0-d0: 20.00
p0-d1: 15.00
p0-d2: 0.00
p1-d0: 0.00
p1-d1: 10.00
p1-d2: 15.00
Costo mínimo total = 255.0


<hr>

## C05E2 — Transbordo por un hub

### Situación completa

* AyudaFederal planifica el traslado semanal de 50 lotes de insumos desde dos depósitos hacia dos regiones, con posibilidad de pasar por un centro de consolidación. 

* El depósito Oeste O dispone de 30 lotes y Norte N de 20. Centro C y Sur S requieren exactamente 25 lotes cada uno. 

* El hub H no produce ni consume: todo lo que entra debe salir. Solo existen seis rutas dirigidas. 

* O-H cuesta 2 por lote y admite 25;
* N-H cuesta 3 y admite 20;
* O-C cuesta 5 y admite 20;
* N-S cuesta 4 y admite 20;
* H-C cuesta 1 y admite 25;
* H-S cuesta 2 y admite 25.

* Los costos se miden en centenas de unidades monetarias y las capacidades en lotes por semana.

* La coordinadora logística decide un flujo continuo por arco para minimizar el costo, respetando capacidades y conservación en cada nodo.

* Los balances son +30 para O, +20 para N, 0 para H y -25 para C y S bajo la convención salida menos entrada.

* Explicitar la convención evita invertir signos y permite auditar el hub independientemente de la matriz.

### Pregunta de decisión

¿Qué flujo debe circular por cada una de las seis rutas para abastecer C y S al menor costo sin violar capacidades ni crear unidades en H?

In [53]:
solver = pywraplp.Solver.CreateSolver("GLOP")

# 50 lotes de insumos

variables = [
    ['O-H', 2, 25],
    ['N-H', 3, 20],
    ['O-C', 5, 20],
    ['N-S', 4, 20],
    ['H-C', 1, 25],
    ['H-S', 2, 25],
]

variables = pd.DataFrame(variables, columns=['arco', 'costo', 'capacidad'])

x = {}
for idx, row in variables.iterrows():
    x[row['arco']] = solver.NumVar(0, row['capacidad'], f"x_{row['arco']}")

# Restricciones:
solver.Add(x['O-H'] + x['N-H'] == x['H-C'] + x['H-S'], "hub")

# O: supply = +30
solver.Add(x['O-H'] + x['O-C'] == 30, "O")

# N: supply = +20
solver.Add(x['N-H'] + x['N-S'] == 20, "N")

# C: demand = -25
solver.Add(x['O-C'] + x['H-C'] == 25, "C")

# S: demand = -25
solver.Add(x['N-S'] + x['H-S'] == 25, "S")

# Función objetivo:
objetivo = (
    x['O-H'] * variables.loc[0, 'costo'] +
    x['N-H'] * variables.loc[1, 'costo'] +
    x['O-C'] * variables.loc[2, 'costo'] +
    x['N-S'] * variables.loc[3, 'costo'] +
    x['H-C'] * variables.loc[4, 'costo'] +
    x['H-S'] * variables.loc[5, 'costo']
)
solver.Minimize(objetivo)

# Print
status = solver.Solve()
es_optimo = status == pywraplp.Solver.OPTIMAL
print("Viable:", es_optimo)

if es_optimo:
    print("Solución óptima encontrada:")
    for arco in x:
        print(f"{arco}: {x[arco].solution_value():.2f}")
    print("Costo mínimo total =", solver.Objective().Value())

Viable: True
Solución óptima encontrada:
O-H: 25.00
N-H: 0.00
O-C: 5.00
N-S: 20.00
H-C: 20.00
H-S: 5.00
Costo mínimo total = 185.0


<hr>

## C05E3 — Demanda no atendida con penalización

### Situación completa

* SaludRed enfrenta una contingencia en la misma red de dos plantas y tres centros.

* P0 puede ofrecer hasta 35 lotes y P1 hasta 25.

* D0 mantiene una demanda de 20, D1 aumenta de 25 a 40 y D2 requiere 15, de modo que la demanda total es 75 frente a solo 60 lotes disponibles.

* Los costos por lote siguen siendo [[4,6,8],[5,4,3]] para las rutas desde P0 y P1 hacia D0, D1 y D2.

* Como resulta imposible cubrir todo, la responsable de Abastecimiento permite registrar r_j>=0, lotes no atendidos en cada centro.

* El balance exige flujo recibido más faltante igual a la demanda; r_j no borra una restricción, sino que hace visible una decisión de nivel de servicio.

* Cada lote faltante recibe una penalización común que se ensaya en 10, 50 y 200 unidades monetarias equivalentes.

* Deben informarse por separado costo logístico, penalización y costo total.

* La localización del faltante importa para los centros, pero una penalización idéntica para todos no expresa prioridades sanitarias diferentes; cualquier regla de equidad o criticidad tendría que acordarse explícitamente.

### Pregunta de decisión

¿Cómo distribuir los 60 lotes disponibles y en qué centros registrar faltantes para minimizar costo logístico más penalización en cada uno de los tres valores ensayados?

In [54]:
variables = [
    ['p0-d0', 4, 35, 20],
    ['p0-d1', 6, 35, 25],
    ['p0-d2', 8, 35, 15],
    ['p1-d0', 5, 25, 20],
    ['p1-d1', 4, 25, 25],
    ['p1-d2', 3, 25, 15],
]

variables = pd.DataFrame(variables, columns=['arco', 'costo', 'oferta', 'demanda'])

variables

,arco,costo,oferta,demanda
0,p0-d0,4,35,20
1,p0-d1,6,35,25
2,p0-d2,8,35,15
3,p1-d0,5,25,20
4,p1-d1,4,25,25
5,p1-d2,3,25,15


In [55]:
# Contraints:
solver = pywraplp.Solver.CreateSolver("GLOP")

# Variables:
# Definición de variables de decisión para cada arco
x = {}
for idx, row in variables.iterrows():
    x[row['arco']] = solver.NumVar(0, row['oferta'], f"x_{row['arco']}")

# Variables de faltante
r = {}
r['d0'] = solver.NumVar(0, 20, "r_d0")
r['d1'] = solver.NumVar(0, 40, "r_d1")
r['d2'] = solver.NumVar(0, 15, "r_d2")

# Restricciones:
solver.Add(x['p0-d0'] + x['p0-d1'] + x['p0-d2'] <= 35, "planta 0")
solver.Add(x['p1-d0'] + x['p1-d1'] + x['p1-d2'] <= 25, "planta 1")

solver.Add(x['p0-d0'] + x['p1-d0'] + r['d0'] == 20, "demanda 0")
solver.Add(x['p0-d1'] + x['p1-d1'] + r['d1'] == 40, "demanda 1")
solver.Add(x['p0-d2'] + x['p1-d2'] + r['d2'] == 15, "demanda 2")

penalizacion = 10 # Cada lote faltante recibe una penalización común que se ensaya en 10, 50 y 200 unidades monetarias equivalentes.

# Función objetivo:
solver.Minimize(
    x['p0-d0'] * variables.loc[0, 'costo'] +
    x['p0-d1'] * variables.loc[1, 'costo'] +
    x['p0-d2'] * variables.loc[2, 'costo'] +
    x['p1-d0'] * variables.loc[3, 'costo'] +
    x['p1-d1'] * variables.loc[4, 'costo'] +
    x['p1-d2'] * variables.loc[5, 'costo'] +
    penalizacion * (r['d0'] + r['d1'] + r['d2'])
)

# Print
status = solver.Solve()
es_optimo = status == pywraplp.Solver.OPTIMAL
print("Viable:", es_optimo)

if es_optimo:
    print("Solución óptima encontrada:")
    for arco in x:
        print(f"{arco}: {x[arco].solution_value():.2f}")
    print("Costo mínimo total =", solver.Objective().Value())

    print(f"r_d0: {r['d0'].solution_value():.2f}")
    print(f"r_d1: {r['d1'].solution_value():.2f}")
    print(f"r_d2: {r['d2'].solution_value():.2f}")

Viable: True
Solución óptima encontrada:
p0-d0: 20.00
p0-d1: 15.00
p0-d2: 0.00
p1-d0: 0.00
p1-d1: 10.00
p1-d2: 15.00
Costo mínimo total = 405.0
r_d0: 0.00
r_d1: 15.00
r_d2: 0.00


<hr>

## C05E4 — Flujo de costo mínimo con una fuente y un sumidero

Modelar y resolver la red de clase:

* demanda 10; arcos `(s,A),(s,B),(A,B),(A,t),(B,t)`;
* costos `2,4,1,5,2`; capacidades `6,8,3,6,8`.
* Antes de pedir código al agente, escribir los cuatro balances nodales y explicar por qué solo `s` puede generar flujo y solo `t` puede absorberlo.
* Entregar tabla por arco, costo reconstruido y controles de balance/capacidad.


In [56]:
variables = [
    ['s-A', 6, 2],
    ['s-B', 8, 4],
    ['A-B', 3, 1],
    ['A-t', 6, 5],
    ['B-t', 8, 2],
]

variables = pd.DataFrame(variables, columns=['arista', 'capacidad', 'costo'])

variables

,arista,capacidad,costo
0,s-A,6,2
1,s-B,8,4
2,A-B,3,1
3,A-t,6,5
4,B-t,8,2


In [57]:
# Contraints:
solver = pywraplp.Solver.CreateSolver("GLOP")

# Variables:
x = {}
for idx, row in variables.iterrows():
    x[row['arista']] = solver.NumVar(0, row['capacidad'], f"x_{row['arista']}")

# Restricciones:
solver.Add(x['s-A'] + x['s-B'] == 10, "nodo s")
solver.Add(-x['s-A'] + x['A-B'] + x['A-t'] == 0, "nodo A")
solver.Add(-x['s-B'] - x['A-B'] + x['B-t'] == 0, "nodo B")
solver.Add(-x['A-t'] - x['B-t'] == -10, "nodo t")

# Función objetivo:
solver.Minimize(x['s-A'] * variables.loc[0, 'costo'] + x['s-B'] * variables.loc[1, 'costo'] + x['A-B'] * variables.loc[2, 'costo'] + x['A-t'] * variables.loc[3, 'costo'] + x['B-t'] * variables.loc[4, 'costo'])

# Print
status = solver.Solve()
es_optimo = status == pywraplp.Solver.OPTIMAL
print("Viable:", es_optimo)

if es_optimo:
    print("Solución óptima encontrada:")
    for arista in x:
        print(f"{arista}: {x[arista].solution_value():.2f}")
    print("Costo mínimo total =", solver.Objective().Value())

Viable: True
Solución óptima encontrada:
s-A: 5.00
s-B: 5.00
A-B: 3.00
A-t: 2.00
B-t: 8.00
Costo mínimo total = 59.0
